# 韩国半导体出口原始公告来源资格

仅核对两个官网 HTML/PDF 附件的月份、美元额、同比、登记日和 PDF 禁发时间，并通过一次 Qlib 日历读出保守美股可用日。来源资格不是收益预测或交易许可。

In [ ]:
from pathlib import Path
import os, json, hashlib, re, io
import pandas as pd
from bs4 import BeautifulSoup
from pypdf import PdfReader
os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE", "true")
root=Path.cwd().resolve()
while not (root/"SPEC.md").is_file():
    if root.parent==root:raise RuntimeError("workspace root missing")
    root=root.parent
card=root/"research/korea_semiconductor_exports_source_20261022"
plan=json.loads((card/"qualification_plan.json").read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan["frozen"] and sha(card/"plan.json")==plan["source_plan_sha256"]
assert sha(card/"retrieval_receipts.json")==plan["html_receipts_sha256"]
assert sha(card/"attachment_plan.json")==plan["attachment_plan_sha256"]
assert sha(card/"attachment_receipts.json")==plan["pdf_receipts_sha256"]
assert not (card/"run_state.json").exists(),"one frozen source qualification attempt already started"
htmls={x["observation_month"]:x for x in json.loads((card/"retrieval_receipts.json").read_text())["queries"]}
pdfs={x["observation_month"]:x for x in json.loads((card/"attachment_receipts.json").read_text())["queries"]}
observed=[];full_text={}
for month,expected in plan["expected_sample"].items():
    page=htmls[month];pdf=pdfs[month]
    for item in (page,pdf):
        path=root/item["relative_path"]
        assert item["http_status"]==200 and sha(path)==item["sha256"]==plan["snapshots_sha256"][item["relative_path"]]
    soup=BeautifulSoup((root/page["relative_path"]).read_bytes(),"html.parser")
    links=[li for li in soup.find_all("li") if li.find("em") and li.find("em").get_text(" ",strip=True)=="Registration date"]
    assert len(links)==1
    registered=links[0].find("span").get_text(" ",strip=True)
    assert registered==expected["registration_date"]
    htmltext=" ".join(soup.stripped_strings)
    reader=PdfReader(io.BytesIO((root/pdf["relative_path"]).read_bytes()))
    pdftext=" ".join(re.sub(r"\s+"," ",(x.extract_text() or "")).strip() for x in reader.pages)
    assert expected["embargo_text"] in pdftext
    amount=expected["semiconductor_exports_usd_billion"]
    yoy=expected["yoy_pct"]
    if month=="2025-05":
        pdf_pattern=r"Semiconductors hit.{0,100}\$13\.8 billion \(up 21\.2 percent\)"
        html_pattern=pdf_pattern
    else:
        assert "Semiconductor exports" in (reader.pages[0].extract_text() or "")
        pdf_pattern=r"reached \$37\.2 billion, up 169\.4 percent"
        html_pattern=r"Semiconductor exports reached \$37\.2 billion, up 169\.4 percent"
    assert re.search(pdf_pattern,pdftext) and re.search(html_pattern,htmltext)
    full_text[month]={"html":htmltext,"pdf":pdftext}
    observed.append({"observation_month":month,"registration_date":registered,
                     "PDF_embargo_not_before_KST":expected["embargo_text"],
                     "semiconductor_exports_USD_billion":amount,"YoY_pct":yoy,
                     "PDF_pages":len(reader.pages),"HTML_SHA256":page["sha256"],"PDF_SHA256":pdf["sha256"]})
import qlib
from qlib.data import D
from qlib.workflow import R
provider=root/plan["calendar_provider"]
assert sha(provider/"qlib_store_manifest.json")==plan["calendar_manifest_sha256"]
assert sha(provider/"calendars/day.txt")==plan["calendar_file_sha256"]
qlib.init(provider_uri=str(provider),region="us",kernels=1)
calendar=pd.DatetimeIndex(D.calendar(start_time="2025-06-01",end_time="2026-06-10",freq="day")).normalize()
for row in observed:
    after=calendar[calendar>pd.Timestamp(row["registration_date"])]
    assert len(after)
    row["first_conservative_US_session_after_registration"]=str(after[0].date())
result={"status":"two_original_release_samples_semantically_qualified_PIT_gap",
        "samples":observed,"HTML_PDF_value_match":True,"native_calendar_rows":len(calendar),
        "PDF_embargo_is_not_actual_release_receipt":True,"registration_may_lag_actual_announcement":True,
        "historical_original_vintage_qualified":False,"continuous_monthly_coverage_qualified":False,
        "model_fits":0,"accounts":0,"market_outcome_reads":0,"D_calendar_calls":1,
        "boundary":"Two original official PDF+HTML samples confirm month/category/currency/value and a not-before embargo, but current downloads do not certify historical first publication or revisions. Conservative rule uses next US session after page registration."}
(card/"run_state.json").write_text(json.dumps({"status":"started","plan_sha256":sha(card/"qualification_plan.json")})+chr(10))
tracking=card/"qlib_recorder"
with R.start(experiment_name="korea_semiconductor_exports_source_20261022",uri=str(tracking)):
    rec=R.get_recorder()
    rec.save_objects(artifact_path="sources",qualification_plan=plan,
                     html_receipts=htmls,pdf_receipts=pdfs,extracted_text=full_text)
    rec.save_objects(artifact_path="analysis",source_qualification=result)
    rec.log_metrics(original_sample_months=len(observed),html_pdf_pairs_matched=2,
                    source_first_public_qualified=0,model_fits=0,accounts=0)
    result.update(recorder_id=rec.id,experiment_id=str(rec.experiment_id),
                  recorder_uri=str(tracking),recorder_local_dir=str(tracking/str(rec.experiment_id)/rec.id))
artifacts=Path(result["recorder_local_dir"])/"artifacts"
verification={"recorder_id":result["recorder_id"],"entries":[{"path":str(p.relative_to(card)),
    "sha256":sha(p),"bytes":p.stat().st_size} for p in sorted(artifacts.rglob("*")) if p.is_file()]}
(card/"verification_manifest.json").write_text(json.dumps(verification,indent=2)+chr(10))
(card/"source_qualification_result.json").write_text(json.dumps(result,indent=2)+chr(10))
summary={"experiment":"korea_semiconductor_exports_source_20261022","status":result["status"],
         "decision":"two_samples_qualified_broader_vintage_and_coverage_required_before_information_screen",
         "result":"source_qualification_result.json","recorder_id":result["recorder_id"],
         "experiment_id":result["experiment_id"],"recorder_uri":result["recorder_uri"],
         "recorder_local_dir":result["recorder_local_dir"],"model_fits":0,"accounts":0,
         "qualified_forward_days":0,"objective_achieved":False,
         "artifact_policy":json.loads((card/"plan.json").read_text())["artifact_policy"]}
(card/"summary.json").write_text(json.dumps(summary,indent=2)+chr(10))
(card/"conclusion.md").write_text("# 中文结论\n\n韩国主管部门官网原始 HTML 与随附 PDF 两份样本的半导体出口口径吻合：2025 年 5 月 USD 13.8B、同比 +21.2%，网页登记 2025-06-02；2026 年 5 月 USD 37.2B、同比 +169.4%，登记 2026-06-01。PDF 首页均标示 6 月 1 日 11:00 KST 前禁发，但它不是实际首发收据。按更保守的网页登记日之后首个 Qlib 美股交易日，分别为 "+observed[0]["first_conservative_US_session_after_registration"]+" 与 "+observed[1]["first_conservative_US_session_after_registration"]+"。Recorder `"+result["recorder_id"]+"`。\n\n这只证明两个月、两年 PDF/HTML 原始数值与语义；历史各月覆盖、修订和首次发布时间尚未合格。没有读取 SOXX 收益、拟合或账户，更没有跑赢 SOXX 的证据。下一步须限定并核查连续月度原始发布，再冻结一个与 SIA 全球三月平均销售额不同的固定信息检验。\n")
(card/"run_state.json").write_text(json.dumps({"status":"completed","recorder_id":result["recorder_id"],"plan_sha256":sha(card/"qualification_plan.json")})+chr(10))
print(json.dumps({"status":result["status"],"samples":observed,"recorder_id":result["recorder_id"]}))

## 中文结论

两份 PDF/HTML 样本数值一致，但首次实际公开版本与连续月度覆盖仍待核查。Recorder 和资格边界见同卡 `conclusion.md`。